# Advanced RNN Topics: Reading and Writing Sequences

A recurrent neural network can do two fundamentally different things with a sequence. It can **read** a sequence and reduce it to a single decision, which we call classification, or it can **write** a sequence one token at a time, which we call generation. Both tasks rely on the same underlying machinery: the RNN processes the sequence one element at a time while maintaining a hidden state that summarizes everything it has seen so far. The difference between the two tasks lies in what we do with that hidden state once the sequence has been processed.

Nearly every topic in this notebook is a variation on one of these two jobs. The material builds up in the following order:

- **Reading a sequence:** using an RNN as a feature extractor to assign a single label to an entire sequence, as in sentiment analysis or spam detection.
- **Word embeddings:** replacing sparse one-hot inputs with dense, trainable vectors that capture relationships between words.
- **Text generation:** training an RNN to produce a sequence one token at a time, along with the challenges this introduces.
- **Decoding and sampling:** once a model is trained, the separate question of how to select tokens from it to produce text.
- **Encoder–decoder models:** combining two RNNs to transform one sequence into another, as in machine translation.
- **Attention:** allowing the decoder to refer back to the entire input rather than relying on a single summary vector.

Each section pairs the concept with runnable TensorFlow and PyTorch examples so that the same idea can be seen in both frameworks.

## Reading a Sequence

When the input is a sequence — a sentence, an email, or a stretch of sensor readings — and the goal is to assign a single label to the whole thing, an RNN is a natural choice. The general idea is to use the RNN as a **feature extractor**. It processes the sequence one element at a time, and at each step it folds the new element into its hidden state, a fixed-size vector that serves as a running summary of everything seen so far. By the time the RNN reaches the end of the sequence, that final hidden state has been shaped by the entire input, giving us a compact, fixed-length representation of a variable-length sequence. A small classifier then turns that representation into a decision.

It is worth noting why this is useful. Reviews and emails arrive in widely varying lengths, but a classifier requires a fixed number of inputs. The RNN bridges this gap: regardless of how long the input is, the hidden state it produces is always the same size.

Two common examples illustrate the idea:

- **Sentiment analysis:** read a movie review and output whether it is *positive* or *negative*.
- **Spam detection:** read an email and label it as *spam* or *not spam*.

In both cases the input is a sequence and the output is a single label that summarizes the entire input.

### Model Architecture Overview

The model has two components, and they are trained together.

1. **The featurizer (the RNN):** It reads the input one step at a time, updating its hidden state as it goes. For classification we discard the intermediate states and keep only the **final** hidden state, treating it as a learned representation of the whole sequence.
2. **The classifier (the head):** This final vector is passed to a small feed-forward layer (a single `Dense` or `Linear` layer is usually sufficient), whose softmax produces one probability per class. To obtain a prediction, we take the `argmax`: the single most likely class.

Because the two components are trained end to end, the RNN does not learn just any summary. It learns the summary that makes the classifier's task as easy as possible.

### Example: Sentiment Classification

We will train on a small corpus of one-line reviews, each labeled positive or negative. To keep this first version simple, every word is converted into a **one-hot vector**: a vector as long as the vocabulary, with a single 1 in the position for that word and 0 everywhere else. A review then becomes a sequence of these one-hot vectors, which is exactly what the RNN reads. The dataset is deliberately small so that everything trains in a few seconds, so the resulting accuracy should be read as a check that the wiring works rather than as a meaningful benchmark.

### Data Preparation

Before either framework can train, we must convert the raw text into integers. This cell builds a vocabulary from the training reviews, maps every word to an id, and pads each review to a common length so that they can be stacked into a single array. Two ids are reserved in advance: `0` for padding and `1` for any unknown word the model encounters at test time. Both the TensorFlow and PyTorch examples below reuse the variables created here.

In [1]:
import numpy as np

# A tiny, real labeled corpus of one-line "reviews". 0 = negative, 1 = positive.
corpus = [
    ("the movie was great and i loved it", 1),
    ("a wonderful film with brilliant acting", 1),
    ("i really enjoyed this fantastic story", 1),
    ("an amazing and delightful experience", 1),
    ("the acting was superb and inspiring", 1),
    ("a beautiful touching and clever film", 1),
    ("i loved the great characters and story", 1),
    ("truly fantastic and wonderful from start to finish", 1),
    ("this film was brilliant and exciting", 1),
    ("a great and enjoyable movie i loved", 1),
    ("the movie was boring and i hated it", 0),
    ("a terrible film with awful acting", 0),
    ("i really disliked this dull story", 0),
    ("an awful and disappointing experience", 0),
    ("the acting was poor and uninspiring", 0),
    ("a boring slow and confusing film", 0),
    ("i hated the weak characters and story", 0),
    ("truly terrible and boring from start to finish", 0),
    ("this film was awful and dull", 0),
    ("a bad and disappointing movie i hated", 0),
]

texts  = [t for t, _ in corpus]
labels = np.array([y for _, y in corpus], dtype=np.int64)

# Build a vocabulary from the training text. Reserve 0 = <pad>, 1 = <unk>.
PAD, UNK = 0, 1
words = sorted({w for t in texts for w in t.split()})
word2idx = {"<pad>": PAD, "<unk>": UNK}
for w in words:
    word2idx[w] = len(word2idx)
vocab_size = len(word2idx)
seq_len = max(len(t.split()) for t in texts)   # pad everything to the longest review

def encode(text):
    '''Turn a sentence into a fixed-length list of token ids (post-padded with <pad>).'''
    ids = [word2idx.get(w, UNK) for w in text.split()][:seq_len]
    return ids + [PAD] * (seq_len - len(ids))

X_idx = np.array([encode(t) for t in texts], dtype=np.int64)   # (num_samples, seq_len)
y = labels

# Held-out sentences we'll actually predict on, using words the model has seen.
test_sentences = [
    "i loved this wonderful film",       # expect positive
    "a boring and terrible movie",       # expect negative
    "the acting was brilliant",          # expect positive
    "i hated the dull story",            # expect negative
]
X_test_idx = np.array([encode(t) for t in test_sentences], dtype=np.int64)
label_name = {0: "NEGATIVE", 1: "POSITIVE"}

print(f"vocab_size = {vocab_size}, seq_len = {seq_len}, num_samples = {len(texts)}")
print("encoded 'the acting was brilliant':", encode("the acting was brilliant"))

vocab_size = 51, seq_len = 8, num_samples = 20
encoded 'the acting was brilliant': [41, 3, 47, 11, 0, 0, 0, 0]


### TensorFlow Example

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(0)
np.random.seed(0)

# Represent each word as a one-hot vector, so each review is a (seq_len, vocab_size) matrix.
X_oh      = tf.one_hot(X_idx, depth=vocab_size)       # (num_samples, seq_len, vocab_size)
X_test_oh = tf.one_hot(X_test_idx, depth=vocab_size)

model = models.Sequential([
    # SimpleRNN reads the sequence of one-hot vectors and returns its final hidden state.
    layers.SimpleRNN(24, input_shape=(seq_len, vocab_size)),
    # Classifier head: a softmax over the two sentiment classes.
    layers.Dense(2, activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit(X_oh, y, epochs=60, batch_size=8, verbose=0)
train_acc = model.evaluate(X_oh, y, verbose=0)[1]
print(f"Training accuracy on the toy corpus: {train_acc:.2f}\n")

# Real predictions: map the argmax of the softmax back to a human-readable label.
probs = model.predict(X_test_oh, verbose=0)
for sent, p in zip(test_sentences, probs):
    cls = int(np.argmax(p))           # argmax of the class softmax -> predicted class
    label = f'"{sent}"'
    print(f"{label:<32}  -->  {label_name[cls]}  (p={p[cls]:.2f})")

/opt/anaconda3/envs/Teaching-TensorFlow/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training accuracy on the toy corpus: 1.00

"i loved this wonderful film"     -->  POSITIVE  (p=0.99)
"a boring and terrible movie"     -->  NEGATIVE  (p=0.91)
"the acting was brilliant"        -->  POSITIVE  (p=0.98)
"i hated the dull story"          -->  NEGATIVE  (p=1.00)


### PyTorch Example

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_classes):
        super().__init__()
        self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True)
        self.fc  = nn.Linear(hidden_size, num_classes)

    def forward(self, x_onehot):                 # x_onehot: (batch, seq_len, vocab_size)
        out, _ = self.rnn(x_onehot)              # out: (batch, seq_len, hidden_size)
        last = out[:, -1, :]                     # final hidden state = sequence summary
        return self.fc(last)                     # raw logits over the classes

model = SentimentRNN(vocab_size, hidden_size=24, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

X = F.one_hot(torch.tensor(X_idx), num_classes=vocab_size).float()
yt = torch.tensor(y)

model.train()
for epoch in range(60):
    optimizer.zero_grad()
    logits = model(X)
    loss = criterion(logits, yt)
    loss.backward()
    optimizer.step()
print(f"Final training loss: {loss.item():.4f}\n")

model.eval()
with torch.no_grad():
    Xt = F.one_hot(torch.tensor(X_test_idx), num_classes=vocab_size).float()
    probs = F.softmax(model(Xt), dim=1)
    for sent, p in zip(test_sentences, probs):
        cls = int(torch.argmax(p))               # argmax of the class softmax -> predicted class
        label = f'"{sent}"'
        print(f"{label:<32}  -->  {label_name[cls]}  (p={p[cls]:.2f})")

### Additional Considerations

#### Variants in Representing the Sequence

Keeping the final hidden state is the most common approach, but it relies heavily on the end of the sequence. Two common alternatives summarize the sequence differently:

- **Mean pooling:** average the hidden states across all time steps, so that every position contributes equally. This tends to smooth out the representation.
- **Max pooling:** take the element-wise maximum across time, which captures the strongest signal wherever it appeared.

(With an LSTM or GRU, the per-step outputs would be pooled in the same way.)

#### End-to-End Training

The loss is computed on the classifier's output and backpropagated through the classifier and into the RNN. As a result, the RNN does not merely summarize the sequence; it learns to summarize it in whatever way makes the final classification easier. The featurizer and the classifier adapt together.

#### Practical Considerations

- **Choice of RNN variant:** vanilla RNNs struggle with long-range dependencies, so LSTMs or GRUs are usually preferred.
- **Regularization and overfitting:** dropout, early stopping, and additional data all help with overfitting, which is a real concern on a corpus this small.
- **Padding and masking:** sequences of different lengths are padded to a common length, and masking prevents the model from treating the padding as meaningful input.

## Word Embeddings

The one-hot encoding from the previous section works, but it forces the model to do more work than necessary. Every word is represented by a vector the size of the entire vocabulary, almost entirely zeros, and, more importantly, every pair of distinct words sits the same distance apart. To a one-hot encoder, "great" and "wonderful" are exactly as unrelated as "great" and "terrible." The model receives no prior information about meaning; it must learn how words relate entirely from scratch through the RNN.

A **word embedding** replaces that one-hot vector with a short, dense, trainable vector. An embedding layer is essentially a lookup table with one row per word in the vocabulary, where each row is a vector of, for example, 16 numbers. Feeding in a word id simply selects its corresponding row. Because those rows are model parameters, they are adjusted during training, and words that the task treats similarly are pulled toward similar vectors. Instead of a sparse on/off indicator, the RNN therefore receives an input that already carries a usable notion of similarity.

### How It Works
- **Lookup:** each word id indexes into the table and retrieves its dense vector. A seven-word review becomes seven vectors of length `embedding_dim` rather than seven vectors of length `vocab_size`.
- **Contextualization:** the RNN processes those vectors and incorporates the surrounding words, so the same word can be represented differently depending on what preceded it.
- **Learning:** gradients flow back through the lookup and adjust each word's row, so the vectors are tuned for the task at hand (sentiment, in this case).

Nothing else about the architecture changes. `one-hot → SimpleRNN` simply becomes `Embedding → SimpleRNN`, and the inputs the RNN reads are both smaller and far more informative. The PyTorch example below also inspects the learned vectors to confirm that related words ended up close together.

### TensorFlow Example

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(0)

embedding_dim = 16

model = models.Sequential([
    # Embedding layer: dense, learned word vectors in place of fixed one-hot vectors.
    # mask_zero=True tells Keras to ignore <pad>=0.
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True),
    layers.SimpleRNN(24),
    layers.Dense(2, activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit(X_idx, y, epochs=60, batch_size=8, verbose=0)   # note: feed token ids directly now
print(f"Training accuracy: {model.evaluate(X_idx, y, verbose=0)[1]:.2f}\n")

probs = model.predict(X_test_idx, verbose=0)
for sent, p in zip(test_sentences, probs):
    cls = int(np.argmax(p))
    label = f'"{sent}"'
    print(f"{label:<32}  -->  {label_name[cls]}  (p={p[cls]:.2f})")

/opt/anaconda3/envs/Teaching-TensorFlow/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Training accuracy: 1.00

"i loved this wonderful film"     -->  NEGATIVE  (p=0.96)
"a boring and terrible movie"     -->  POSITIVE  (p=0.53)
"the acting was brilliant"        -->  POSITIVE  (p=0.55)
"i hated the dull story"          -->  NEGATIVE  (p=0.64)


### PyTorch Example

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class EmbeddingSentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc  = nn.Linear(hidden_size, num_classes)

    def forward(self, x_ids):                    # x_ids: (batch, seq_len) integer tokens
        emb = self.embedding(x_ids)              # (batch, seq_len, embedding_dim)
        out, _ = self.rnn(emb)
        return self.fc(out[:, -1, :])

model = EmbeddingSentimentRNN(vocab_size, embedding_dim=16, hidden_size=24, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

X = torch.tensor(X_idx)
yt = torch.tensor(y)
model.train()
for epoch in range(60):
    optimizer.zero_grad()
    loss = criterion(model(X), yt)
    loss.backward()
    optimizer.step()
print(f"Final training loss: {loss.item():.4f}\n")

model.eval()
with torch.no_grad():
    probs = F.softmax(model(torch.tensor(X_test_idx)), dim=1)
    for sent, p in zip(test_sentences, probs):
        cls = int(torch.argmax(p))
        label = f'"{sent}"'
        print(f"{label:<32}  -->  {label_name[cls]}  (p={p[cls]:.2f})")

    # Did the embedding learn anything? Compare cosine similarity of a few words.
    E = model.embedding.weight                    # (vocab_size, embedding_dim)
    def sim(a, b):
        va, vb = E[word2idx[a]], E[word2idx[b]]
        return F.cosine_similarity(va, vb, dim=0).item()
    print("\nLearned word-vector cosine similarities (task-tuned, tiny data):")
    print(f"  loved   ~ wonderful : {sim('loved', 'wonderful'):+.2f}")
    print(f"  loved   ~ hated     : {sim('loved', 'hated'):+.2f}")
    print(f"  boring  ~ dull      : {sim('boring', 'dull'):+.2f}")

### Weight Tying

The embedding layer maps a word id to a vector on the way *in*. A model that generates text requires the opposite mapping on the way *out*: a final layer that takes the hidden state and produces one score per word in the vocabulary, which the softmax then converts into the probability of each word being the next one. There is a natural symmetry between these two layers. The embedding is a `(vocab_size × embedding_dim)` matrix, and the output layer is a `(hidden_size × vocab_size)` matrix. When `embedding_dim` equals `hidden_size`, the output layer has exactly the shape of the embedding, transposed.

**Weight tying** is a technique that takes advantage of this symmetry by using a single shared matrix for both directions instead of learning two separate ones. Three considerations make it worthwhile:

- **Parameter efficiency:** for a realistic vocabulary of tens of thousands of words, the embedding and the output layer are usually the two largest matrices in the entire model. Sharing them roughly halves that cost.
- **Regularization:** a single matrix performing both roles has less capacity to overfit than two independent ones, which tends to improve generalization.
- **Consistency:** a word is represented the same way whether it is being read in or written out. This is a reasonable property to expect of a language model, and in practice it reliably improves results.

## Text Generation

Reading a sequence produced a single label. Writing is the complementary task: the model generates a sequence one token at a time, and each token it produces is fed back in as part of the input for the next step. This feedback loop is what makes the process **auto-regressive**, meaning that every prediction is conditioned on everything generated so far. Expressed as probabilities, the model factors the probability of an entire sequence into a product of next-token predictions:

$$
P(x_1, x_2, \dots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, x_2, \dots, x_{t-1})
$$

This single equation actually contains two distinct questions, and keeping them separate avoids a good deal of confusion:

1. **Training:** how do we fit these per-step conditional probabilities to data?
2. **Generation:** once the model is trained, how do we actually select tokens from it to produce text?

This section addresses training. Selecting tokens is the subject of the later section on decoding and sampling.

## Teacher Forcing

To train a generator, we need, at each step, the context so far together with the correct next token. The obvious approach is to let the model build its own context: predict the first token, feed it back, predict the second token, and so on. Early in training, however, the model's predictions are unreliable, so the second step is conditioned on an incorrect first token, the third step on context built from earlier mistakes, and the errors accumulate. The model never gets a clean opportunity to learn what we actually want it to learn: given correct context, predict the next token.

**Teacher forcing** is a training strategy that addresses this problem by feeding the model the *ground-truth* previous token at each step during training, rather than its own prediction. Every step now sees the correct context, the gradient signal stays clean, and training is faster and more stable. This is exactly what the shifted-by-one targets in the data-preparation code below set up: the input is the true text, and the target is that same text shifted one position to the right.

The drawback appears at generation time, and it has a name: **exposure bias**. When the model finally runs on its own, there is no ground truth to feed it, so it must consume its own imperfect outputs — a setting it never encountered during training. A single early mistake can lead it into territory it never practiced on. This gap between how the model is trained (always-correct context) and how it generates (self-produced context) is one of the reasons the decoding choices in the next section matter as much as they do.

<img src="teacher-forcing.png" alt="Teacher forcing: feeding the ground-truth token as the next input during training" width="340" height="582">

## Vanishing and Exploding Gradients

Training a generator over a long sequence runs into a specific obstacle, and it is the reason gated architectures such as LSTMs and GRUs were developed in the first place.

To train an RNN, we unroll it through time and backpropagate through every step, a procedure called **backpropagation through time**. The difficulty is that the *same* recurrent weight matrix $W$ is applied at every step, so when the gradient flows back across $t$ steps it accumulates roughly $t$ factors of $W$ multiplied together. Repeatedly multiplying by the same matrix is numerically unstable, and the direction it takes depends on the magnitude of $W$ (loosely, its largest eigenvalue).

#### Problems
- **Vanishing gradients:** if those repeated factors are smaller than 1, the product shrinks toward zero before the gradient reaches the early steps. The model cannot connect a later output back to an early input, so it fails to learn long-range dependencies — the beginning of a long review has no measurable effect on the loss.
- **Exploding gradients:** if the factors are larger than 1, the product grows without bound, producing large, unstable updates and `NaN` values.

#### Solutions
- **Gated architectures:** this is the primary remedy. LSTMs and GRUs add an additive memory path along with gates that decide what to keep and what to overwrite, allowing gradients to travel across many steps without the same multiplicative decay. This is why they are the default whenever sequences become long.
- **Gradient clipping:** rescales the gradient whenever its norm exceeds a threshold. It is a simple, direct remedy for the exploding case.
- **Normalization:** keeps the scale of activations and gradients within a reasonable range; layer normalization is better suited to variable-length sequences than batch normalization.
- **Activation choice:** plays a smaller role; non-saturating activations such as ReLU vanish less readily than `tanh` or sigmoid in some settings.

## Greedy And Beam Search - Decoding Strategies

When generating sequences, such as in machine translation or text generation, selecting the right decoding strategy is crucial for generating high-quality outputs. Two common strategies are **Greedy Decoding** and **Beam Search**. Both approaches aim to determine the best sequence of words based on the model's probability estimates, but they differ in how they explore the space of possible sequences.

### Greedy Decoding

**Greedy decoding** is the simplest method for sequence generation. 

It operates by making a series of local, one-step optimal decisions. At each time step, the algorithm looks at the probability distribution over the next possible words and selects the one with the highest probability. This decision is made without considering how the choice might influence future selections.

Greedy decoding effectively builds a search tree where only one branch is followed — the one corresponding to the highest probability word at each step.

Imagine a search tree where each node represents a possible word in the sequence. Greedy decoding only follows one branch — the one that seems best at the current step — ignoring all alternative paths that might lead to a better overall sequence. 

While this method is computationally efficient and straightforward to implement, its narrow focus on the immediate best choice often leads to outputs that are predictable and lack variety. In many cases, this leads to repetitive sequences that do not capture the full richness of the language.

Due to its limitations, greedy decoding is not commonly used in practice for tasks that require nuanced or varied outputs, although it can be useful in simpler or more constrained settings.


### Beam Search

**Beam search** is a more sophisticated decoding strategy that addresses some of the limitations of greedy decoding.

Instead of choosing only the best word at each time step, beam search keeps track of the top **K** candidate sequences, where **K** is known as the beam width. This allows the algorithm to explore several possible sequences simultaneously.

The beam width (typically set between 5 and 10 in practice) controls the number of hypotheses maintained at each step. A larger beam width allows for a more exhaustive search, increasing the likelihood of finding a better overall sequence, but at the cost of increased computational complexity.

#### Process Overview
  1. **Initialization:**  
     Start with an initial token (often a start-of-sequence token) and initialize the beam with this starting point.
  2. **Expansion:**  
     At each time step, expand all candidate sequences in the beam by appending all possible next words and computing their cumulative probabilities.
  3. **Pruning:**  
     Retain only the top **K** sequences based on their cumulative probabilities.
  4. **Termination:**  
     Continue the expansion and pruning steps until all sequences in the beam reach an end-of-sequence token or a predetermined maximum length.

#### Application
  Beam search is particularly popular in tasks like machine translation (e.g., translating English to German) where generating a grammatically coherent and contextually accurate sentence is critical. By considering multiple candidate sequences, beam search often produces more natural and varied outputs compared to greedy decoding.

### Summary

Both greedy decoding and beam search are used to generate sequences from probabilistic models:
- **Greedy Decoding:**  
  - Fast and simple.
  - Locally optimal but often suboptimal overall.
  - Tends to produce generic and repetitive outputs.
- **Beam Search:**  
  - More computationally expensive but explores multiple candidate sequences.
  - Achieves a better balance between quality and diversity in the output.
  - Commonly used in complex sequence generation tasks like machine translation.

Selecting the appropriate decoding strategy depends on the specific task requirements and the trade-off between computational resources and the quality of generated sequences.

<img src="greedy-beam.jpeg" alt="Greedy keeps one branch; beam search keeps the top-K branches at each step" width="720" height="379">

## Advanced Sampling Strategies

Always selecting the highest probability word at every step (as in greedy decoding) typically produces sentences that are grammatically correct but can be overly predictable and repetitive. 

This occurs because the model continually picks the most common choices, leading to generic output. In many applications — such as creative text generation or dialogue systems — more interesting outputs are desired. To achieve this, advanced sampling strategies introduce controlled randomness to balance between quality and diversity. 

Two important factors come into play:

- **Quality:** Favoring high-probability words to maintain coherence.
- **Diversity:** Allowing less probable words a chance to be selected, enriching the output with variety.

### Top-K Sampling

Top-K sampling is a straightforward extension of greedy decoding. Instead of selecting only the single highest probability word, the algorithm proceeds as follows:

1. **Candidate Selection:**  
   At each time step, pre-select the top **k** most likely words based on the model's probability distribution.

2. **Truncation and Renormalization:**  
   Truncate the full probability distribution to include only these top-k candidates, then renormalize the probabilities so that they sum to one.

3. **Random Sampling:**  
   Randomly sample the next word from this reduced distribution according to the renormalized probabilities.

This strategy strikes a balance between maintaining a high quality (by limiting choices to the most likely words) and introducing diversity (by allowing a random selection among them). 

However, because **k** is fixed, its effectiveness can vary depending on the nature of the original probability distribution. In some cases, the distribution might be very peaked, so the top-k choices cover nearly all the probability mass; in other cases, a flat distribution means that even the top-k words represent only a small fraction of the total mass.

### Top-P Sampling (Nucleus Sampling)

Top-P sampling, also known as nucleus sampling, dynamically adjusts the candidate pool based on the cumulative probability rather than a fixed number of words. 

The process is as follows:

1. **Cumulative Probability Threshold:**  
   At each step, sort all candidate words by their probability and select the smallest set of words whose cumulative probability exceeds a predetermined threshold **p**.

2. **Renormalization and Sampling:**  
   Renormalize this subset of candidates to form a valid probability distribution and sample the next word from this nucleus.

By focusing on a threshold rather than a fixed count, top-P sampling adapts to the probability distribution's shape. For peaked distributions, the nucleus might be very small, while for flatter distributions, it will be larger. This flexibility ensures that only the most contextually relevant words are considered, balancing coherence with the potential for creative variations.

### Temperature Sampling

Temperature sampling modifies the overall probability distribution by scaling the logits (pre-softmax scores) before applying the softmax function. 

The process involves:

1. **Logit Scaling:**  
   Divide the logits by a temperature factor before computing the softmax. This adjustment reshapes the probability distribution.

2. **Effect of Temperature:**  
   - **Low Temperature (< 1):**  
     The distribution becomes sharper; high-probability words become even more dominant, reducing randomness. This tends to favor high-quality, coherent outputs but can result in repetitive text.
   - **High Temperature (> 1):**  
     The distribution flattens, increasing the chance of selecting lower-probability words, which introduces more diversity and creativity into the generated text.

Temperature sampling allows for a smooth trade-off between quality and diversity without discarding any part of the original probability distribution.

## Sampling Examples

The following cells give concrete, framework-native implementations of the three sampling strategies — top-k, top-p, and temperature — applied to a set of dummy logits, first in TensorFlow and then in PyTorch.

### TensorFlow Example

#### Import Libraries

In [ ]:
import tensorflow as tf
import numpy as np

#### Top-K Sampling

In [ ]:
def sample_top_k_tf(logits, k=10):
    """
    Performs top-K sampling:
      1. Select the top k logits.
      2. Mask out others by setting them to a very low value.
      3. Renormalize and sample.
    """
    
    logits = tf.convert_to_tensor(logits)
    topk = tf.math.top_k(logits, k=k)
    topk_logits = topk.values
    topk_indices = topk.indices

    # Create a mask for the top-k values
    full_mask = tf.fill(tf.shape(logits), float('-inf'))
    mask = tf.tensor_scatter_nd_update(full_mask, tf.expand_dims(topk_indices, 1), topk_logits)
    probs = tf.nn.softmax(mask)
    sample = tf.random.categorical(tf.math.log([probs]), num_samples=1)
    
    return tf.gather(tf.range(tf.shape(logits)[0]), tf.squeeze(sample, axis=0))

#### Top-P / Nucleus Sampling

In [ ]:
def sample_top_p_tf(logits, p=0.9):
    """
    Performs top-P (nucleus) sampling:
      1. Sort logits and compute probabilities.
      2. Determine the minimal set where cumulative probability exceeds p.
      3. Mask out others, renormalize, and sample.
    """
    
    logits = tf.convert_to_tensor(logits)
    
    # Sort logits in descending order
    sorted_logits, sorted_indices = tf.math.top_k(logits, k=tf.shape(logits)[0])
    sorted_probs = tf.nn.softmax(sorted_logits)
    cumulative_probs = tf.math.cumsum(sorted_probs)
    
    # Create a mask: keep tokens where cumulative probability is less than p
    mask = cumulative_probs <= p
    
    # Ensure at least one token is kept
    mask = tf.concat([[True], mask[1:]], axis=0)
    
    # Set logits for tokens not in the nucleus to a very low value
    masked_logits = tf.where(mask, sorted_logits, tf.fill(tf.shape(sorted_logits), float('-inf')))
    new_probs = tf.nn.softmax(masked_logits)
    sample = tf.random.categorical(tf.math.log([new_probs]), num_samples=1)
    
    # Map back to original indices
    return tf.gather(sorted_indices, tf.squeeze(sample, axis=0))

#### Temperature Sampling

In [ ]:
def sample_with_temperature_tf(logits, temperature=1.0):
    """
    Scale logits by temperature and sample one token.
    """
    
    scaled_logits = logits / temperature
    probs = tf.nn.softmax(scaled_logits)
    
    # tf.random.categorical expects a 2D tensor; we expand dims and squeeze the output
    sample = tf.random.categorical(tf.math.log([probs]), num_samples=1)
    return tf.squeeze(sample, axis=0)

#### Usage

In [ ]:
# Example usage with dummy logits
vocab_size = 50
dummy_logits_tf = tf.random.normal([vocab_size])  # Example logits for 50 tokens

# Temperature sampling example
temp_sample_tf = sample_with_temperature_tf(dummy_logits_tf, temperature=0.8)
print("Temperature Sampled token index (TF):", temp_sample_tf.numpy())

# Top-K sampling example
topk_sample_tf = sample_top_k_tf(dummy_logits_tf, k=10)
print("Top-K Sampled token index (TF):", topk_sample_tf.numpy())

# Top-P sampling example
topp_sample_tf = sample_top_p_tf(dummy_logits_tf, p=0.9)
print("Top-P Sampled token index (TF):", topp_sample_tf.numpy())

### PyTorch Example

#### Import Libraries

In [ ]:
import torch
import torch.nn.functional as F

#### Top-K Sampling

In [ ]:
def sample_top_k(logits, k=10):
    """
    Performs top-K sampling:
      1. Select the top k logits.
      2. Mask out the rest (set to -infinity).
      3. Renormalize and sample.
    """
    
    # Get the top k logits and their indices
    topk_logits, topk_indices = torch.topk(logits, k)
    
    # Create a mask that sets values not in top-k to -infinity
    mask = torch.full_like(logits, float('-inf'))
    mask[topk_indices] = topk_logits
    
    # Apply softmax to get a valid probability distribution
    probs = F.softmax(mask, dim=0)
    
    return torch.multinomial(probs, num_samples=1)

#### Top-P / Nucleus Sampling

In [ ]:
def sample_top_p(logits, p=0.9):
    """
    Performs top-P (nucleus) sampling:
      1. Sort logits and compute softmax probabilities.
      2. Determine the minimal set of tokens where the cumulative probability exceeds p.
      3. Mask out tokens outside this set, renormalize, and sample.
    """
    
    # Sort logits in descending order
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=0)
    
    # Compute cumulative probabilities
    cumulative_probs = torch.cumsum(sorted_probs, dim=0)
    
    # Determine which tokens to keep (ensure at least one token is kept)
    cutoff = cumulative_probs > p
    cutoff[0] = False  # Always keep the top token
    
    # Mask out tokens beyond the nucleus
    sorted_logits[cutoff] = float('-inf')
    
    # Renormalize the masked logits
    probs = F.softmax(sorted_logits, dim=0)
    sample = torch.multinomial(probs, num_samples=1)
    
    # Map the sample back to the original indices
    return sorted_indices[sample]

#### Temperature Sampling

In [ ]:
def sample_with_temperature(logits, temperature=1.0):
    """
    Scale logits by temperature and sample one token.
    Lower temperatures (<1) sharpen the distribution, higher temperatures (>1) flatten it.
    """
    scaled_logits = logits / temperature
    probs = F.softmax(scaled_logits, dim=0)
    # Multinomial sampling from the probability distribution
    return torch.multinomial(probs, num_samples=1)

#### Usage

In [ ]:
# Example usage with dummy logits
vocab_size = 50
dummy_logits = torch.randn(vocab_size)  # Example logits for 50 tokens

# Temperature sampling example
temp_sample = sample_with_temperature(dummy_logits, temperature=0.8)
print("Temperature Sampled token index:", temp_sample.item())

# Top-K sampling example
topk_sample = sample_top_k(dummy_logits, k=10)
print("Top-K Sampled token index:", topk_sample.item())

# Top-P sampling example
topp_sample = sample_top_p(dummy_logits, p=0.9)
print("Top-P Sampled token index:", topp_sample.item())


## A Shared Generator to Decode From

In the previous week, you trained a character-level RNN to generate Shakespeare. Here we train a similar but much smaller model on a handful of review-style sentences, and then run every decoding strategy on that same model so that each one produces real text. In this section the model itself is not the focus; what matters is what we do with its per-step distribution.

We use a character-level model because the vocabulary is small — only a few dozen characters — so it trains on a CPU in seconds and the per-step distribution is small enough to reason about. Because the model is small, the output will be review-flavored text rather than Shakespeare. The contrast between strategies on a fixed model, however, holds regardless of how good the model is: greedy decoding is deterministic and somewhat repetitive, while sampling is more varied.

We build the model in both PyTorch and TensorFlow, and each one exposes the same simple interface, `logits_for(context)`, which returns a vector of next-character scores. This allows a single set of strategy functions to drive either model, since decoding operates on the distribution rather than on the framework that produced it.

### The Training Corpus and Char Vocabulary (Run Once)

In [3]:
import numpy as np

# A small corpus of review-style lines for the char-level model.
gen_text = (
    "the movie was great and i really loved it. "
    "the film was wonderful and the acting was brilliant. "
    "i loved the story and the characters were great. "
    "a wonderful and exciting film that i really enjoyed. "
    "the movie was boring and i did not like it. "
    "the film was terrible and the acting was awful. "
) * 8   # repeat so the tiny model sees enough structure

chars = sorted(set(gen_text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
gen_vocab = len(chars)
GEN_SEQ = 24            # context window the model is trained on
print(f"char vocab size = {gen_vocab}, corpus length = {len(gen_text)} chars")
print("chars:", "".join(chars))

# Build (context -> next char) training pairs.
data_ids = np.array([stoi[c] for c in gen_text], dtype=np.int64)
Xg, Yg = [], []
for i in range(len(data_ids) - GEN_SEQ):
    Xg.append(data_ids[i:i + GEN_SEQ])
    Yg.append(data_ids[i + 1:i + GEN_SEQ + 1])   # next-char targets (teacher forcing!)
Xg = np.array(Xg, dtype=np.int64)
Yg = np.array(Yg, dtype=np.int64)
print("training pairs:", Xg.shape, "->", Yg.shape)

char vocab size = 25, corpus length = 2320 chars
chars:  .abcdefghijklmnorstuvwxy
training pairs: (2296, 24) -> (2296, 24)


### PyTorch Model

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

class CharRNN(nn.Module):
    def __init__(self, vocab, emb=24, hidden=96):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb)
        self.rnn   = nn.RNN(emb, hidden, batch_first=True)
        self.head  = nn.Linear(hidden, vocab)

    def forward(self, x):                       # x: (batch, seq)
        out, _ = self.rnn(self.embed(x))        # (batch, seq, hidden)
        return self.head(out)                   # (batch, seq, vocab) -- logits at every step

torch_model = CharRNN(gen_vocab)
opt = torch.optim.Adam(torch_model.parameters(), lr=0.005)
lossf = nn.CrossEntropyLoss()

Xt, Yt = torch.tensor(Xg), torch.tensor(Yg)
torch_model.train()
for epoch in range(40):
    opt.zero_grad()
    logits = torch_model(Xt)                                  # (N, seq, vocab)
    loss = lossf(logits.reshape(-1, gen_vocab), Yt.reshape(-1))
    loss.backward()
    opt.step()
print(f"PyTorch char-RNN final loss: {loss.item():.3f}")

torch_model.eval()
def torch_logits_for(context):
    '''The uniform interface: a context string -> numpy vector of next-char logits.'''
    ids = [stoi.get(c, 0) for c in context][-GEN_SEQ:]        # last GEN_SEQ chars
    with torch.no_grad():
        out = torch_model(torch.tensor([ids]))                # (1, len, vocab)
    return out[0, -1].numpy()                                 # logits for the next char

### TensorFlow Model

In [4]:
import tensorflow as tf

tf.random.set_seed(0)

tf_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(gen_vocab, 24),
    tf.keras.layers.SimpleRNN(96, return_sequences=True),     # logits at every step
    tf.keras.layers.Dense(gen_vocab),                         # raw logits (from_logits=True below)
])
tf_model.compile(optimizer=tf.keras.optimizers.Adam(0.005),
                 loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
tf_model.fit(Xg, Yg, epochs=40, batch_size=64, verbose=0)
print(f"TF char-RNN final loss: {tf_model.evaluate(Xg, Yg, verbose=0):.3f}")

def tf_logits_for(context):
    '''Same interface as the PyTorch model: context string -> next-char logits (numpy).'''
    ids = [stoi.get(c, 0) for c in context][-GEN_SEQ:]
    out = tf_model(np.array([ids]))                           # (1, len, vocab)
    return out[0, -1].numpy()

TF char-RNN final loss: 0.125


### The Strategy Functions

Each function takes a `logits_for` callable (the interface exposed by either model) along with a prompt, and returns the generated text. The same code works on either model, since both return the same object: a distribution over the vocabulary. Below are NumPy implementations of greedy decoding, beam search, temperature, top-k, and top-p.

In [8]:
import numpy as np

def softmax(logits):
    logits = np.asarray(logits, dtype=np.float64)
    z = logits - logits.max()
    e = np.exp(z)
    return e / e.sum()

# ---------- deterministic, likelihood-seeking ----------
def greedy_generate(logits_for, prompt, n=80):
    text = prompt
    for _ in range(n):
        text += itos[int(np.argmax(logits_for(text)))]   # argmax every step == greedy
    return text

def beam_generate(logits_for, prompt, n=80, width=5):
    beams = [(prompt, 0.0)]                                # (text, cumulative log-prob)
    for _ in range(n):
        candidates = []
        for text, score in beams:
            logp = np.log(softmax(logits_for(text)) + 1e-12)
            for idx in np.argsort(logp)[-width:]:         # only the most promising extensions
                candidates.append((text + itos[int(idx)], score + logp[idx]))
        candidates.sort(key=lambda tc: tc[1], reverse=True)
        beams = candidates[:width]                        # prune to the beam width
    return beams[0][0]                                    # highest-scoring sequence

# ---------- stochastic, diversity-seeking ----------
def _sample_idx(probs, rng):
    return int(rng.choice(len(probs), p=probs / probs.sum()))

def temperature_generate(logits_for, prompt, n=80, T=1.0, seed=0):
    rng, text = np.random.default_rng(seed), prompt
    for _ in range(n):
        text += itos[_sample_idx(softmax(logits_for(text) / T), rng)]
    return text

def top_k_generate(logits_for, prompt, n=80, k=5, seed=0):
    rng, text = np.random.default_rng(seed), prompt
    for _ in range(n):
        logits = np.asarray(logits_for(text), dtype=np.float64).copy()
        kth = np.partition(logits, -k)[-k]                # k-th largest logit
        logits[logits < kth] = -np.inf                    # drop everything below the top-k
        text += itos[_sample_idx(softmax(logits), rng)]
    return text

def top_p_generate(logits_for, prompt, n=80, p=0.9, seed=0):
    rng, text = np.random.default_rng(seed), prompt
    for _ in range(n):
        logits = np.asarray(logits_for(text), dtype=np.float64)
        probs = softmax(logits)
        order = np.argsort(probs)[::-1]                   # high -> low
        cum = np.cumsum(probs[order])
        keep = order[:np.searchsorted(cum, p) + 1]        # smallest set with cum prob >= p
        masked = np.full_like(logits, -np.inf)
        masked[keep] = logits[keep]
        text += itos[_sample_idx(softmax(masked), rng)]
    return text
print("strategy functions ready")

strategy functions ready


### TensorFlow Model

Nothing about the strategies changes; we simply pass `tf_logits_for` instead of `torch_logits_for`. The two models learn slightly different weights, so the exact text differs, but each strategy behaves in the same way.

In [10]:
prompt = "the movie was "
print("=== TensorFlow model (identical strategy code) ===\n")
print("GREEDY            :", repr(greedy_generate(tf_logits_for, prompt)))
print("BEAM (width=5)     :", repr(beam_generate(tf_logits_for, prompt, width=5)))
print("TEMPERATURE T=1.0  :", repr(temperature_generate(tf_logits_for, prompt, T=1.0)))
print("TOP-P p=0.9        :", repr(top_p_generate(tf_logits_for, prompt, p=0.9)))

=== TensorFlow model (identical strategy code) ===

GREEDY            : 'the movie was boring and i did not like it. the film was terrible and the acting was awful. th'
BEAM (width=5)     : 'the movie was boring and i did not like it. the film was terrible and the acting was awful. th'
TEMPERATURE T=1.0  : 'the movie was great and i really loved it. the film was wonderful and the acting was brilliant'
TOP-P p=0.9        : 'the movie was great and i really loved it. the film was wonderful and the acting was brilliant'


### PyTorch Model

Compare the outputs below. Greedy decoding and beam search are deterministic, so running them again produces identical output, and they tend to settle on safe, repetitive phrasing. The sampling methods are random and noticeably more varied. Notice how higher temperature and wider top-k or top-p settings progressively loosen the text and eventually degrade it.

In [ ]:
prompt = "the movie was "
print("=== PyTorch model ===\n")
print("GREEDY            :", repr(greedy_generate(torch_logits_for, prompt)))
print("BEAM (width=5)     :", repr(beam_generate(torch_logits_for, prompt, width=5)))
print()
print("TEMPERATURE T=0.5  :", repr(temperature_generate(torch_logits_for, prompt, T=0.5)))
print("TEMPERATURE T=1.0  :", repr(temperature_generate(torch_logits_for, prompt, T=1.0)))
print("TEMPERATURE T=1.5  :", repr(temperature_generate(torch_logits_for, prompt, T=1.5)))
print()
print("TOP-K k=3          :", repr(top_k_generate(torch_logits_for, prompt, k=3)))
print("TOP-K k=10         :", repr(top_k_generate(torch_logits_for, prompt, k=10)))
print("TOP-P p=0.9        :", repr(top_p_generate(torch_logits_for, prompt, p=0.9)))

### Why Beam Search Can Outperform Greedy Decoding

A slightly worse first token can still lead to a better overall sequence. This small, hand-constructed "model" (no training is involved) makes that concrete. Greedy decoding selects the best token at the first step and becomes stuck, while a beam of width 2 keeps the alternative in consideration and ultimately arrives at the higher-probability sequence.

In [11]:
import numpy as np

# A 2-token toy vocabulary {"A", "B"} with a hand-set distribution.
def toy_logits_for(seq):
    # seq is the generated string so far (just A's and B's here)
    if seq.endswith("B"):    return np.log([0.05, 0.95])   # after B, B is almost certain
    if seq.endswith("A"):    return np.log([0.50, 0.50])   # after A, it's a coin flip
    return np.log([0.55, 0.45])                            # first step slightly favors A

toy_itos = {0: "A", 1: "B"}

def toy_greedy(steps):
    seq = ""
    for _ in range(steps):
        seq += toy_itos[int(np.argmax(toy_logits_for(seq)))]
    return seq

def toy_beam(steps, width=2):
    beams = [("", 0.0)]
    for _ in range(steps):
        cand = []
        for seq, score in beams:
            logp = toy_logits_for(seq)
            for idx in (0, 1):
                cand.append((seq + toy_itos[idx], score + logp[idx]))
        cand.sort(key=lambda x: x[1], reverse=True)
        beams = cand[:width]
    return beams[0]

g = toy_greedy(2)
b_seq, b_logp = toy_beam(2, width=2)
def seqprob(s):
    p, cur = 1.0, ""
    for ch in s:
        p *= np.exp(toy_logits_for(cur))[0 if ch == "A" else 1]; cur += ch
    return p
print(f"greedy chose {g!r}  (overall prob {seqprob(g):.3f})")
print(f"beam   chose {b_seq!r}  (overall prob {seqprob(b_seq):.3f})")
print("-> beam found the more probable sequence greedy's tunnel vision missed.")

greedy chose 'AA'  (overall prob 0.275)
beam   chose 'BB'  (overall prob 0.427)
-> beam found the more probable sequence greedy's tunnel vision missed.


---
## Encoder-Decoder Architecture

So far, a single RNN has handled the entire task — either reading one sequence or writing one sequence. Many problems, however, require mapping *one sequence to another*: translating English to French, summarizing an article, or transcribing speech to text. In these problems the input and output can have different lengths and even different orderings, so reading-then-classifying or writing-from-scratch is not sufficient on its own.

The **encoder–decoder** framework is a foundational approach designed for exactly this situation, converting one sequence into another. It is widely used in sequence-to-sequence learning and has been applied to tasks including language translation, text summarization, and speech recognition.

The architecture divides the overall task into two components:
 - **Encoder:** processes the entire input sequence and compresses it into a compact representation, often called the context vector.
 - **Decoder:** uses this context vector to generate the output sequence, one token at a time.

![](https://miro.medium.com/v2/resize:fit:1400/1*1JcHGUU7rFgtXC_mydUA_Q.jpeg)

### How It Works

**Encoding:**

The encoder reads the input sequence, processing it token by token. At each step, it updates its internal state to capture the context and semantics of the input. The final hidden state of the encoder is then treated as a distilled summary of the entire input sequence. This summary is expected to capture all relevant information needed for generating the output.

**Decoding:**

Once the encoder has produced the context vector, the decoder begins generating the output sequence. Starting with an initial state (often derived from the context vector), the decoder predicts the first output token. It then uses that token, along with its current state and the context vector, to predict the next token, and so on. This process continues until a special end-of-sequence token is generated, signaling that the output is complete.


### Strengths and Weaknesses

**Strengths:**
 - **Modularity:** By separating the encoding and decoding processes, the model can be more flexible and easier to adapt to different tasks.
 - **Applicability:** The architecture is versatile and has been successfully applied to numerous tasks involving sequence transformations.
    

**Limitations:**
  - **Information Bottleneck:** Compressing an entire input sequence into a single fixed-length context vector can result in the loss of fine-grained details, particularly for long or complex inputs.
  - **Sequential Dependency:** Traditional implementations often rely on sequential processing (using RNNs, for example), which can make it challenging to capture long-range dependencies in the input.
    

The encoder–decoder model set the stage for subsequent innovations in sequence modeling. While early implementations often used Recurrent Neural Networks (RNNs) for both the encoder and decoder, the fundamental idea of transforming one sequence into another has inspired a range of advanced architectures. These include models that incorporate attention mechanisms to alleviate the information bottleneck and, eventually, the development of transformer architectures that further improve on both performance and scalability.

---
## Encoder-Decoder RNNs

**The encoder–decoder model with RNNs is a specific implementation of the general encoder–decoder framework, where both the encoder and decoder are built using recurrent neural networks.** This configuration leverages the sequential processing capabilities of RNNs to capture the temporal dynamics inherent in language, speech, and other sequential data.

In traditional sequence-to-sequence tasks, the model must transform an input sequence into an output sequence, such as converting a sentence in one language to another. RNNs naturally handle sequential data by updating their hidden state with each new token. However, when RNNs are used in isolation, they can struggle with variable-length sequences and long-range dependencies. 

The encoder–decoder structure with RNNs was designed to address these challenges:

**Encoder RNN:**
- **Function:** Processes the input sequence token by token.
- **Mechanism:** At each time step, the encoder updates its hidden state using the current token and the previous hidden state. This recursive process allows the network to build an internal representation of the entire sequence.
- **Output:** The final hidden state acts as a compressed summary (context vector) of the entire input sequence.
- **Benefit:** Captures sequential patterns and dependencies in the input, albeit in a fixed-length vector.


**Decoder RNN:**
- **Function:** Generates the output sequence based on the encoded representation.
- **Mechanism:** Starting from the context vector, the decoder predicts the output token at each time step, using its previous outputs and hidden state to generate the next token.
- **Output:** A sequence of tokens that represents the target sequence, such as a translated sentence.
- **Benefit:** Provides a structured way to generate sequences while maintaining context across time steps.

### Mathematical Formulation

#### Encoder

Given an input sequence $ X = \{x_1, x_2, \dots, x_T\} $, the encoder processes each token step-by-step using a recurrent formula:
$$
h_t = f(W_{xh} \, x_t + W_{hh} \, h_{t-1} + b_h)
$$
- $ h_t $: Hidden state at time $ t $
- $ W_{xh} $ and $ W_{hh} $: Weight matrices
- $ b_h $: Bias term
- $ f $: Activation function (e.g., $\tanh$, ReLU)

The final hidden state $ h_T $ serves as the **context vector**:
$$
c = h_T
$$

#### Decoder

The decoder RNN generates the output sequence $ Y = \{y_1, y_2, \dots, y_{T'}\} $ conditioned on the context vector $ c $. Its recurrence is:
$$
s_t = f(W_{ys} \, y_{t-1} + W_{cs} \, c + W_{ss} \, s_{t-1} + b_s)
$$

- $ s_t $: Decoder hidden state at time $ t $
- $ y_{t-1} $: Previously generated output (usually embedded)
- $ W_{ys} $, $ W_{cs} $, $ W_{ss} $: Weight matrices
- $ b_s $: Bias term

The output token is computed as:
$$
y_t = \text{softmax}(W_{out} \, s_t + b_{out})
$$

<p align="center">
  <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTBvrDTed79lFHC8GMLQ757v11n_Y1nV0V_1Q&s" width="70%"/>
</p>


### Limitations with This Approach

While encoder–decoder RNNs were a significant breakthrough, they come with inherent limitations:

- **Fixed-Length Context Vector:**
Compressing an entire input sequence into a single vector may result in loss of important details, particularly for long or complex sequences. This fixed-size bottleneck can limit performance when the input contains intricate or diverse information.
- **Sequential Bottleneck in Decoding:**
Because the decoder relies on sequential prediction, the generation process can be slow and may struggle with maintaining long-term dependencies over extended outputs.

These challenges led to the development of advanced techniques like attention mechanisms, which allow the decoder to dynamically refer back to different parts of the input sequence during generation, thereby alleviating the information bottleneck of a fixed-length context vector.

### TensorFlow Example

#### Import Libraries

In [12]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, GRU, Dense
from tensorflow.keras.models import Model
import numpy as np

#### Define the Encoder

In [13]:
class Encoder(Model):
    def __init__(self, vocab_size, embedding_dim, enc_units):
        super(Encoder, self).__init__()
        self.enc_units = enc_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        # return_sequences=True to get outputs at all time steps
        # return_state=True to get the final hidden state
        self.gru = GRU(enc_units, return_sequences=True, return_state=True)
    
    def call(self, x, hidden):
        # x: (batch_size, sequence_length)
        x = self.embedding(x)  # (batch_size, sequence_length, embedding_dim)
        output, state = self.gru(x, initial_state=hidden)
        return output, state
    
    def initialize_hidden_state(self, batch_size):
        return tf.zeros((batch_size, self.enc_units))

#### Define the Decoder

In [14]:
class Decoder(Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.gru = GRU(dec_units, return_sequences=True, return_state=True)
        self.fc = Dense(vocab_size)
    
    def call(self, x, hidden):
        # x: (batch_size, 1) as we process one token at a time
        x = self.embedding(x)  # (batch_size, 1, embedding_dim)
        output, state = self.gru(x, initial_state=hidden)
        # reshape output from (batch_size, 1, dec_units) to (batch_size, dec_units)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)  # (batch_size, vocab_size)
        return x, state

#### Set Up Training Parameters and Data

In [15]:
vocab_inp_size = 10   # Input vocabulary size
vocab_tar_size = 10   # Target vocabulary size
embedding_dim = 16    # Embedding dimension
units = 16            # Number of GRU units
BATCH_SIZE = 1        # Batch size (for simplicity)
n_epochs = 300        # Number of training epochs
SOS_token = 0         # Start-of-sequence token fed to the decoder first

# Create sample input and target sequences (batch_size=1).
# The model's job is to learn this fixed input -> target mapping.
input_seq = tf.constant([[1, 2, 3, 4, 5]], dtype=tf.int32)   # shape: (1, sequence_length)
target_seq = tf.constant([[1, 2, 3, 4, 6]], dtype=tf.int32)

# Initialize encoder and decoder models
encoder = Encoder(vocab_inp_size, embedding_dim, units)
decoder = Decoder(vocab_tar_size, embedding_dim, units)

# Define the optimizer and the loss function
optimizer = tf.keras.optimizers.SGD(learning_rate=0.05)
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


#### Define Training and Inference Steps

In [16]:
def train_step(input_seq, target_seq, teacher_forcing_ratio=0.5):
    loss = 0

    with tf.GradientTape() as tape:
        # Initialize encoder hidden state and encode the input sequence
        enc_hidden = encoder.initialize_hidden_state(BATCH_SIZE)
        enc_output, enc_hidden = encoder(input_seq, enc_hidden)

        # Set initial decoder state to the encoder's final hidden state
        dec_hidden = enc_hidden
        # Start-of-sequence token
        dec_input = tf.expand_dims([SOS_token] * BATCH_SIZE, 1)  # shape: (batch_size, 1)

        # Iterate over each token in the target sequence
        for t in range(target_seq.shape[1]):
            # Pass the current token and state through the decoder
            predictions, dec_hidden = decoder(dec_input, dec_hidden)
            # Compare the predicted token with the actual target token
            loss += loss_object(target_seq[:, t], predictions)

            # Decide whether to use teacher forcing this step
            if np.random.rand() < teacher_forcing_ratio:
                # Teacher forcing: feed the true target token as the next input
                dec_input = tf.expand_dims(target_seq[:, t], 1)
            else:
                # Otherwise: feed the decoder's own prediction as the next input
                predicted_ids = tf.argmax(predictions, axis=1, output_type=tf.int32)
                dec_input = tf.expand_dims(predicted_ids, 1)

    # Compute the average loss per token, then backpropagate and update weights
    batch_loss = loss / int(target_seq.shape[1])
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))

    return batch_loss


def evaluate(input_seq, target_len):
    """Greedy decoding with no teacher forcing (the honest test of the model).
    Returns the list of predicted token indices."""
    enc_hidden = encoder.initialize_hidden_state(BATCH_SIZE)
    enc_output, enc_hidden = encoder(input_seq, enc_hidden)

    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([SOS_token] * BATCH_SIZE, 1)

    predictions_out = []
    for t in range(target_len):
        predictions, dec_hidden = decoder(dec_input, dec_hidden)
        predicted_id = tf.argmax(predictions, axis=1, output_type=tf.int32)
        predictions_out.append(int(predicted_id[0].numpy()))
        # Feed the model's own prediction as the next input
        dec_input = tf.expand_dims(predicted_id, 1)
    return predictions_out


#### Run Training and Inference

In [17]:
# ---- Training: repeat the step many times so the model can actually learn ----
print("Training...\n")
for epoch in range(1, n_epochs + 1):
    batch_loss = train_step(input_seq, target_seq)
    if epoch == 1 or epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Avg loss per token: {batch_loss.numpy():.4f}")

# ---- Inference: greedy decoding with no teacher forcing ----
predictions = evaluate(input_seq, target_seq.shape[1])
input_list = input_seq.numpy()[0].tolist()
target_list = target_seq.numpy()[0].tolist()

print("\nInput sequence:     ", input_list)
print("Target sequence:    ", target_list)
print("Predicted sequence: ", predictions)

correct = sum(p == t for p, t in zip(predictions, target_list))
print(f"\nTokens correct: {correct}/{len(target_list)}")


Training...

Epoch   1 | Avg loss per token: 2.2976
Epoch  50 | Avg loss per token: 0.8533
Epoch 100 | Avg loss per token: 0.1502
Epoch 150 | Avg loss per token: 0.0361
Epoch 200 | Avg loss per token: 0.0190
Epoch 250 | Avg loss per token: 0.0127
Epoch 300 | Avg loss per token: 0.0094

Input sequence:      [1, 2, 3, 4, 5]
Target sequence:     [1, 2, 3, 4, 6]
Predicted sequence:  [1, 2, 3, 4, 6]

Tokens correct: 5/5


### PyTorch Example

#### Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

#### Define the Encoder

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
    
    def forward(self, input, hidden):
        # Embed the input token and reshape for the GRU
        embedded = self.embedding(input).view(1, 1, self.hidden_size)
        output, hidden = self.gru(embedded, hidden)
        return output, hidden
    
    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size)

#### Define the Decoder

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)
    
    def forward(self, input, hidden):
        # Embed the input token and reshape for the GRU
        embedded = self.embedding(input).view(1, 1, self.hidden_size)
        output, hidden = self.gru(embedded, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

#### Training Loop and Inference

In [ ]:
if __name__ == "__main__":
    # Hyperparameters
    input_size = 10    # vocabulary size for input
    output_size = 10   # vocabulary size for output
    hidden_size = 16
    teacher_forcing_ratio = 0.5
    n_epochs = 300
    SOS_token = 0      # start-of-sequence token fed to the decoder first

    # Example input and target sequences (represented as indices).
    # The model's job is to learn this fixed input -> target mapping.
    input_seq = torch.tensor([1, 2, 3, 4, 5], dtype=torch.long)   # Input sequence of length 5
    target_seq = torch.tensor([1, 2, 3, 4, 6], dtype=torch.long)  # Target sequence of length 5

    encoder = EncoderRNN(input_size, hidden_size)
    decoder = DecoderRNN(hidden_size, output_size)

    # Optimizers and loss function for training
    encoder_optimizer = optim.SGD(encoder.parameters(), lr=0.05)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=0.05)
    criterion = nn.NLLLoss()

    def run_sequence(use_teacher_forcing_allowed):
        """Encode the input, then decode one token at a time.
        Returns the summed loss and the list of predicted token indices."""
        # Encode the input sequence into a single context (the final hidden state)
        encoder_hidden = encoder.initHidden()
        for token in input_seq:
            _, encoder_hidden = encoder(token.unsqueeze(0), encoder_hidden)

        # Initialize the decoder with the start-of-sequence token and the context
        decoder_input = torch.tensor([SOS_token], dtype=torch.long)
        decoder_hidden = encoder_hidden

        loss = 0
        predictions = []
        for target_token in target_seq:
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_token.unsqueeze(0))

            # Record the model's own prediction at this step
            topv, topi = decoder_output.topk(1)
            predictions.append(topi.item())

            # Decide whether to feed the true token (teacher forcing) or the
            # model's own prediction as the next decoder input.
            use_teacher_forcing = (use_teacher_forcing_allowed
                                   and torch.rand(1).item() < teacher_forcing_ratio)
            if use_teacher_forcing:
                decoder_input = target_token.unsqueeze(0)            # true token
            else:
                decoder_input = topi.squeeze().detach().unsqueeze(0)  # own prediction
        return loss, predictions

    # ---- Training: repeat the step many times so the model can actually learn ----
    print("Training...\n")
    for epoch in range(1, n_epochs + 1):
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        loss, _ = run_sequence(use_teacher_forcing_allowed=True)

        loss.backward()
        encoder_optimizer.step()
        decoder_optimizer.step()

        if epoch == 1 or epoch % 50 == 0:
            print(f"Epoch {epoch:3d} | Avg loss per token: {loss.item() / len(target_seq):.4f}")

    # ---- Inference: greedy decoding with no teacher forcing (the honest test) ----
    with torch.no_grad():
        _, predictions = run_sequence(use_teacher_forcing_allowed=False)

    print("\nInput sequence:     ", input_seq.tolist())
    print("Target sequence:    ", target_seq.tolist())
    print("Predicted sequence: ", predictions)

    correct = sum(p == t for p, t in zip(predictions, target_seq.tolist()))
    print(f"\nTokens correct: {correct}/{len(target_seq)}")


## RNNs with Attention

The encoder–decoder model has one significant weakness: the entire input must pass through a single fixed-length context vector. For a short sentence this is adequate, but for a long one the encoder is forced to compress everything into one vector, and earlier details are often crowded out by the time the decoder needs them.

The **attention mechanism** was introduced to remove this bottleneck. Rather than providing the decoder with a single summary vector, attention retains *all* of the encoder's per-step hidden states and, at each decoding step, allows the decoder to construct a *new* context vector tailored to the token it is currently generating. In effect, the decoder is able to look back over the entire input and decide which parts are most relevant at each step. This produces a more flexible model that can better capture long-range dependencies.

### How Attention Works

The mechanism can be understood as a soft, weighted lookup over the encoder states, carried out in three steps at every decoding step $t$.

#### 1. Compute Alignment Scores

We have the decoder's previous hidden state $s_{t-1}$ (a summary of what has been generated so far) and the encoder's hidden state $h_j$ for every input position $j$. For each $h_j$ we compute an *alignment score* $e_{tj}$ that measures how relevant input position $j$ is to the token about to be produced. In Bahdanau attention, the score is computed by a small learned network:

$$
e_{tj} = \text{score}(s_{t-1}, h_j) = v_a^\top \tanh(W_a \, s_{t-1} + U_a \, h_j)
$$

Here $W_a$, $U_a$, and $v_a$ are learnable parameters. $W_a$ and $U_a$ project the decoder state and an encoder state into a shared space, $\tanh$ combines them, and $v_a^\top$ reduces the result to a single number — the relevance of position $j$ at step $t$.

#### 2. Convert Scores into Attention Weights

A raw score is not directly usable, so the scores for all input positions are passed through a softmax. This produces the **attention weights** $\alpha_{tj}$: non-negative values that sum to 1 across the input, forming a distribution that indicates where the decoder should focus.

$$
\alpha_{tj} = \frac{\exp(e_{tj})}{\sum_{k=1}^{T} \exp(e_{tk})}
$$

#### 3. Build the Context Vector

The context vector $c_t$ is the weighted average of the encoder states, using those weights. Positions that the model judges relevant contribute most, while the others are largely ignored. Because the weights are recomputed at every step, $c_t$ changes as the decoder advances through the output; this is the *dynamic* context vector that replaces the single fixed one.

$$
c_t = \sum_{j=1}^{T} \alpha_{tj} \, h_j
$$

Finally, the decoder incorporates this context vector into its usual recurrence, alongside the previous output and its own state:

$$
s_t = f(W_{ys} \, y_{t-1} + W_{cs} \, c_t + W_{ss} \, s_{t-1} + b_s)
$$

The entire mechanism is differentiable, so the scoring network is learned end to end with the rest of the model; the model is never told what to attend to but instead learns it from the data.

### Bahdanau Attention

Bahdanau attention, also known as *additive attention*, was introduced by Bahdanau, Cho, and Bengio in 2014 to address the limitations of standard encoder–decoder models. Traditional sequence-to-sequence architectures relied on a single fixed-length context vector to represent the entire input sequence, which often degraded performance on longer or more complex sequences. Bahdanau attention resolves this by computing a new, dynamic context vector $c_t$ at each decoding step. The term "additive" refers to the score function, which *adds* the projected decoder and encoder states inside the $\tanh$, in contrast to the multiplicative (dot-product) scoring used by later architectures such as the Transformer.

### TensorFlow Example

#### Import Libraries

In [18]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, GRU, Dense, Concatenate
from tensorflow.keras.models import Model
import numpy as np

#### Define the Encoder

In [19]:
class Encoder(Model):
    def __init__(self, vocab_size, embedding_dim, enc_units):
        super(Encoder, self).__init__()
        self.enc_units = enc_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.gru = GRU(enc_units, return_sequences=True, return_state=True)
    
    def call(self, x, hidden):
        x = self.embedding(x)  # (batch, seq_len, embedding_dim)
        output, state = self.gru(x, initial_state=hidden)
        return output, state
    
    def initialize_hidden_state(self, batch_size):
        return tf.zeros((batch_size, self.enc_units))

#### Bahdanau Attention Definition

In [20]:
class BahdanauAttention(Model):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)
    
    def call(self, query, values):
        # query: decoder hidden state at current time step (batch, hidden)
        # values: encoder outputs (batch, seq_len, hidden)
        query_with_time_axis = tf.expand_dims(query, 1)  # (batch, 1, hidden)
        
        # Score: (batch, seq_len, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        
        # Attention weights: (batch, seq_len, 1)
        attention_weights = tf.nn.softmax(score, axis=1)
        
        # Context vector: weighted sum of encoder outputs (batch, hidden)
        context_vector = attention_weights * values  # (batch, seq_len, hidden)
        context_vector = tf.reduce_sum(context_vector, axis=1)
        
        return context_vector, attention_weights

#### Define the Decoder

In [21]:
class Decoder(Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.gru = GRU(dec_units, return_sequences=True, return_state=True)
        self.fc = Dense(vocab_size)
        
        self.attention = BahdanauAttention(dec_units)
    
    def call(self, x, hidden, enc_output):
        # x: (batch, 1) -> current input token
        x = self.embedding(x)  # (batch, 1, embedding_dim)
        # Calculate attention
        context_vector, attention_weights = self.attention(hidden, enc_output)
        context_vector = tf.expand_dims(context_vector, 1)  # (batch, 1, dec_units)
        
        # Concatenate context vector with embedding
        x = Concatenate(axis=-1)([context_vector, x])  # (batch, 1, dec_units+embedding_dim)
        output, state = self.gru(x, initial_state=hidden)
        output = tf.reshape(output, (-1, output.shape[2]))  # (batch, dec_units)
        x = self.fc(output)  # (batch, vocab_size)
        return x, state, attention_weights

#### Training and Inference Example

In [22]:
vocab_inp_size = 10
vocab_tar_size = 10
embedding_dim = 16
units = 16
BATCH_SIZE = 1
n_epochs = 300        # Number of training epochs
SOS_token = 0         # Start-of-sequence token fed to the decoder first

# Sample input and target sequences (batch size = 1).
# The model's job is to learn this fixed input -> target mapping.
input_seq = tf.constant([[1, 2, 3, 4, 5]], dtype=tf.int32)   # (batch, seq_len)
target_seq = tf.constant([[1, 2, 3, 4, 6]], dtype=tf.int32)

encoder = Encoder(vocab_inp_size, embedding_dim, units)
decoder = Decoder(vocab_tar_size, embedding_dim, units)
optimizer = tf.keras.optimizers.SGD(learning_rate=0.05)
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


def train_step(input_seq, target_seq, teacher_forcing_ratio=0.5):
    loss = 0
    with tf.GradientTape() as tape:
        batch_size = input_seq.shape[0]
        enc_hidden = encoder.initialize_hidden_state(batch_size)
        enc_output, enc_hidden = encoder(input_seq, enc_hidden)

        dec_hidden = enc_hidden
        dec_input = tf.expand_dims([SOS_token] * batch_size, 1)  # (batch, 1)

        for t in range(target_seq.shape[1]):
            # The decoder attends over all encoder outputs at each step
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            loss += loss_object(target_seq[:, t], predictions)

            if np.random.rand() < teacher_forcing_ratio:
                # Teacher forcing: feed the true target token as the next input
                dec_input = tf.expand_dims(target_seq[:, t], 1)
            else:
                # Otherwise: feed the decoder's own prediction as the next input
                predicted_ids = tf.argmax(predictions, axis=1, output_type=tf.int32)
                dec_input = tf.expand_dims(predicted_ids, 1)

    batch_loss = loss / int(target_seq.shape[1])
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return batch_loss


def evaluate(input_seq, target_len):
    """Greedy decoding with no teacher forcing. Returns predicted token indices."""
    batch_size = input_seq.shape[0]
    enc_hidden = encoder.initialize_hidden_state(batch_size)
    enc_output, enc_hidden = encoder(input_seq, enc_hidden)

    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([SOS_token] * batch_size, 1)

    predictions_out = []
    for t in range(target_len):
        predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
        predicted_id = tf.argmax(predictions, axis=1, output_type=tf.int32)
        predictions_out.append(int(predicted_id[0].numpy()))
        dec_input = tf.expand_dims(predicted_id, 1)
    return predictions_out


# ---- Training: repeat the step many times so the model can actually learn ----
print("Training...\n")
for epoch in range(1, n_epochs + 1):
    batch_loss = train_step(input_seq, target_seq)
    if epoch == 1 or epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Avg loss per token: {batch_loss.numpy():.4f}")

# ---- Inference: greedy decoding with no teacher forcing ----
predictions = evaluate(input_seq, target_seq.shape[1])
input_list = input_seq.numpy()[0].tolist()
target_list = target_seq.numpy()[0].tolist()

print("\nInput sequence:     ", input_list)
print("Target sequence:    ", target_list)
print("Predicted sequence: ", predictions)

correct = sum(p == t for p, t in zip(predictions, target_list))
print(f"\nTokens correct: {correct}/{len(target_list)}")


Training...

Epoch   1 | Avg loss per token: 2.2998
Epoch  50 | Avg loss per token: 0.8074
Epoch 100 | Avg loss per token: 0.1387
Epoch 150 | Avg loss per token: 0.0436
Epoch 200 | Avg loss per token: 0.0220
Epoch 250 | Avg loss per token: 0.0142
Epoch 300 | Avg loss per token: 0.0104

Input sequence:      [1, 2, 3, 4, 5]
Target sequence:     [1, 2, 3, 4, 6]
Predicted sequence:  [1, 2, 3, 4, 6]

Tokens correct: 5/5


### PyTorch Example

#### Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

#### Define the Encoder

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
    
    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, self.hidden_size)  # (1, batch, hidden_size)
        output, hidden = self.gru(embedded, hidden)
        return output, hidden
    
    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size)

#### Bahdanau Attention Definition

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.hidden_size = hidden_size
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Parameter(torch.rand(hidden_size))
    
    def forward(self, hidden, encoder_outputs):
        # hidden: (1, batch, hidden_size) current decoder hidden state
        # encoder_outputs: (seq_len, batch, hidden_size)
        seq_len = encoder_outputs.size(0)
        batch_size = encoder_outputs.size(1)
        
        # Repeat decoder hidden state seq_len times
        hidden = hidden.repeat(seq_len, 1, 1)  # (seq_len, batch, hidden_size)
        # Concatenate encoder outputs and repeated hidden state along the last dim
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), 2)))  # (seq_len, batch, hidden_size)
        # Compute alignment scores (dot product with v)
        energy = energy.permute(1, 0, 2)  # (batch, seq_len, hidden_size)
        v = self.v.repeat(batch_size, 1).unsqueeze(1)  # (batch, 1, hidden_size)
        scores = torch.bmm(v, energy.permute(0, 2, 1))  # (batch, 1, seq_len)
        attn_weights = torch.softmax(scores, dim=2)  # (batch, 1, seq_len)
        # Compute context vector as weighted sum of encoder_outputs
        encoder_outputs = encoder_outputs.permute(1, 0, 2)  # (batch, seq_len, hidden_size)
        context = torch.bmm(attn_weights, encoder_outputs)  # (batch, 1, hidden_size)
        context = context.permute(1, 0, 2)  # (1, batch, hidden_size)
        return context, attn_weights

#### Define the Decoder

In [ ]:
class DecoderRNNWithAttention(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNNWithAttention, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        # Combine context vector and embedding before feeding into GRU
        self.gru = nn.GRU(hidden_size * 2, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)
    
    def forward(self, input, hidden, encoder_outputs):
        # Embed input token
        embedded = self.embedding(input).view(1, 1, self.hidden_size)
        # Compute attention context vector
        context, attn_weights = self.attention(hidden, encoder_outputs)
        # Concatenate embedded input and context vector
        rnn_input = torch.cat((embedded, context), 2)  # (1, 1, 2*hidden_size)
        output, hidden = self.gru(rnn_input, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden, attn_weights

#### Example Training and Inference Loop

In [ ]:
if __name__ == "__main__":
    # Hyperparameters
    input_size = 10
    output_size = 10
    hidden_size = 16
    teacher_forcing_ratio = 0.5
    n_epochs = 300
    SOS_token = 0      # start-of-sequence token fed to the decoder first

    # Input and target sequences (indices). The model learns this fixed mapping.
    input_seq = torch.tensor([1, 2, 3, 4, 5], dtype=torch.long)    # Length = 5
    target_seq = torch.tensor([1, 2, 3, 4, 6], dtype=torch.long)   # Length = 5

    encoder = EncoderRNN(input_size, hidden_size)
    decoder = DecoderRNNWithAttention(hidden_size, output_size)

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=0.05)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=0.05)
    criterion = nn.NLLLoss()

    def run_sequence(use_teacher_forcing_allowed):
        """Encode the input, then decode with attention one token at a time.
        Returns the summed loss and the list of predicted token indices."""
        # Encode the input sequence, keeping every encoder output for attention
        encoder_hidden = encoder.initHidden()
        encoder_outputs = torch.zeros(len(input_seq), 1, hidden_size)
        for t, token in enumerate(input_seq):
            encoder_output, encoder_hidden = encoder(token.unsqueeze(0), encoder_hidden)
            encoder_outputs[t] = encoder_output[0]

        # Initialize the decoder with the start-of-sequence token and encoder state
        decoder_input = torch.tensor([SOS_token], dtype=torch.long)
        decoder_hidden = encoder_hidden

        loss = 0
        predictions = []
        for target_token in target_seq:
            decoder_output, decoder_hidden, attn_weights = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            loss += criterion(decoder_output, target_token.unsqueeze(0))

            # Record the model's own prediction at this step
            topv, topi = decoder_output.topk(1)
            predictions.append(topi.item())

            use_teacher_forcing = (use_teacher_forcing_allowed
                                   and torch.rand(1).item() < teacher_forcing_ratio)
            if use_teacher_forcing:
                decoder_input = target_token.unsqueeze(0)            # true token
            else:
                decoder_input = topi.squeeze().detach().unsqueeze(0)  # own prediction
        return loss, predictions

    # ---- Training: repeat the step many times so the model can actually learn ----
    print("Training...\n")
    for epoch in range(1, n_epochs + 1):
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        loss, _ = run_sequence(use_teacher_forcing_allowed=True)

        loss.backward()
        encoder_optimizer.step()
        decoder_optimizer.step()

        if epoch == 1 or epoch % 50 == 0:
            print(f"Epoch {epoch:3d} | Avg loss per token: {loss.item() / len(target_seq):.4f}")

    # ---- Inference: greedy decoding with no teacher forcing (the honest test) ----
    with torch.no_grad():
        _, predictions = run_sequence(use_teacher_forcing_allowed=False)

    print("\nInput sequence:     ", input_seq.tolist())
    print("Target sequence:    ", target_seq.tolist())
    print("Predicted sequence: ", predictions)

    correct = sum(p == t for p, t in zip(predictions, target_seq.tolist()))
    print(f"\nTokens correct: {correct}/{len(target_seq)}")


## Conclusion

This notebook traced a single idea — an RNN processing a sequence one step at a time while maintaining a hidden state — through a progression of increasingly capable models.

- **Reading** a sequence treats the RNN as a feature extractor, summarizing a variable-length input into a fixed-size representation that a small classifier can act on.
- **Word embeddings** replace sparse one-hot inputs with dense, trainable vectors, giving the model a usable notion of similarity before training even begins.
- **Writing** a sequence turns the same machinery into an auto-regressive generator, trained efficiently with teacher forcing and kept stable over long sequences by gated architectures that address vanishing and exploding gradients.
- **Decoding and sampling** showed that a trained generator still leaves a design choice open: deterministic strategies (greedy decoding and beam search) aim for the most probable sequence, while stochastic strategies (temperature, top-k, and top-p sampling) trade some coherence for variety.
- **Encoder–decoder models** combine two RNNs to map one sequence to another, and **attention** removes the fixed-length bottleneck by letting the decoder build a fresh, weighted view of the entire input at every step.

The final idea — attention — is the conceptual bridge to the Transformer, which discards recurrence entirely and relies on attention alone. That architecture is the subject of the material that follows.